In [1]:
import pandas as pd, numpy as np
from scipy.spatial import cKDTree
from pyproj import Transformer
from shapely import wkt
from shapely.geometry import LineString, Point
from shapely.strtree import STRtree

bg = pd.read_csv("../data/processed/background_points.csv")   # already has demand/competition/bus/parking

_to_utm = Transformer.from_crs("EPSG:4326", "EPSG:32645", always_xy=True)
def to_xy(lat, lon):
    x, y = _to_utm.transform(np.asarray(lon,float), np.asarray(lat,float)); return np.asarray(x), np.asarray(y)
RADIUS = 500.0

# build only the 3 new engines
inter = pd.read_csv("../data/raw_data/roads/intersections_all_areas.csv").drop_duplicates(subset=["latitude","longitude"])
ix,iy = to_xy(inter.latitude.values, inter.longitude.values); inter_tree = cKDTree(np.c_[ix,iy])

bld = pd.read_csv("../data/processed/buildings_combined.csv")
bx,by = to_xy(bld.latitude.values, bld.longitude.values); bld_tree = cKDTree(np.c_[bx,by])

roads = pd.read_csv("../data/raw_data/roads/roads_all_areas.csv")
main = roads[roads.highway.isin(["primary","trunk","secondary"])]
main_lines = [LineString(zip(*_to_utm.transform(*wkt.loads(w).xy))) for w in main.geometry_wkt]
road_tree = STRtree(main_lines)

# add only the 3 missing columns, using the lat/lon already in the file
X, Y = to_xy(bg.latitude.values, bg.longitude.values)
bg["intersection_count_500m"] = [len(inter_tree.query_ball_point([x,y], RADIUS)) for x,y in zip(X,Y)]
bg["building_count_500m"]     = [len(bld_tree.query_ball_point([x,y], RADIUS)) for x,y in zip(X,Y)]
bg["dist_to_main_road_m"]     = [round(Point(x,y).distance(main_lines[road_tree.nearest(Point(x,y))]),1) for x,y in zip(X,Y)]

bg.to_csv("../data/processed/background_points.csv", index=False)
bg.groupby("search_area")["building_count_500m"].mean().round(0)   # sanity: all non-zero

search_area
Baneshwor                  2592.0
Bhaktapur durbar square    1340.0
Boudha stupa               1776.0
Durbar Marg                2537.0
Kirtipur                   1132.0
Koteshwor                  1884.0
New Road                   2936.0
Patan durbar square        1971.0
Pulchowk                   1873.0
Name: building_count_500m, dtype: float64

In [3]:
import pandas as pd, numpy as np

bg = pd.read_csv("../data/processed/background_points.csv")
bg["avg_restaurant_rating_500m"] = bg["avg_restaurant_rating_500m"].fillna(0)
bg["avg_review_ratings_500m"]    = bg["avg_review_ratings_500m"].fillna(0)

def nb(df, c):   # benefit
    mn, mx = df[c].min(), df[c].max()
    return pd.Series(0.0, index=df.index) if mx == mn else (df[c]-mn)/(mx-mn)
def nc(df, c):   # cost (invert)
    mn, mx = df[c].min(), df[c].max()
    return pd.Series(1.0, index=df.index) if mx == mn else (mx-df[c])/(mx-mn)

# Demand (nested: anchor / daytime / commercial each equal)
anchor  = ["cinema_count_500m","museum_count_500m","temple_count_500m","recreation_count_500m"]
daytime = ["office_count_500m","college_count_500m","school_count_500m","hospital_count_500m","clinic_count_500m"]
commer  = ["retail_count_500m","bank_count_500m"]
A = pd.DataFrame({c: nb(bg,c) for c in anchor}).mean(axis=1)
D = pd.DataFrame({c: nb(bg,c) for c in daytime}).mean(axis=1)
C = pd.DataFrame({c: nb(bg,c) for c in commer}).mean(axis=1)
bg["Demand"] = ((A+D+C)/3).round(4)

# Accessibility (3 benefit + inverted road distance)
acc = pd.DataFrame({"bus": nb(bg,"bus_stop_count_500m"), "park": nb(bg,"parking_space_count_500m"),
                    "inter": nb(bg,"intersection_count_500m"), "road": nc(bg,"dist_to_main_road_m")})
bg["Accessibility"] = acc.mean(axis=1).round(4)

# Population
bg["Population"] = nb(bg,"building_count_500m").round(4)

# Competition (nearest = benefit; count/rating/reviews = cost)
comp = pd.DataFrame({"near": nb(bg,"nearest_restaurant_m"), "cnt": nc(bg,"competitor_count_500m"),
                     "rat": nc(bg,"avg_restaurant_rating_500m"), "rev": nc(bg,"avg_review_ratings_500m")})
bg["Competition"] = comp.mean(axis=1).round(4)

bg[["place_id","latitude","longitude","search_area","Demand","Accessibility","Competition","Population"]] \
    .to_csv("../data/processed/background_features_merged.csv", index=False)

In [4]:
import pandas as pd

rest = pd.read_csv("../data/processed/features_merged.csv")             # restaurants
bg   = pd.read_csv("../data/processed/background_features_merged.csv")  # background

factors = ["Demand", "Accessibility", "Competition", "Population"]
keep = ["place_id", "latitude", "longitude", "search_area"] + factors

rest = rest[keep].copy(); rest["label"] = 1     # real restaurant
bg   = bg[keep].copy();   bg["label"]   = 0     # background point

dataset = pd.concat([rest, bg], ignore_index=True)
dataset.to_csv("../data/processed/dataset_final.csv", index=False)

print(dataset.shape, dict(dataset.label.value_counts()))
dataset.groupby("label")[factors].mean().round(3)

(4172, 9) {0: np.int64(2700), 1: np.int64(1472)}


,Demand,Accessibility,Competition,Population
label,,,,
0,0.219,0.384,0.510,0.352
1,0.269,0.419,0.433,0.387
